# Adaptation Outcomes Metrics

This notebook creates the downloadable **Dry Proofing**, **Relocation**, and **Flood Protection** card CSVs. All cards report baseline, adapted, and avoided economic and social outcomes plus concentration-index change. Flood Protection adds degree-of-urbanisation and design return-period choices, with central, lower, and upper adaptation-cost estimates in millions of USD. Reduction percentages, quintile ratios, and adjusted adaptation costs are intentionally excluded.

## 0. Setup

Load the reusable pipeline code, Kenya configuration, and configured administrative boundaries. ADM0 uses the country ISO3 code when a one-row input omits `shapeID`.

In [ ]:
from pathlib import Path
import importlib
import sys

WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
SRC_DIRECTORY = REPO_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from national_tool_metrics import load_country_config
from national_tool_metrics.boundaries import load_admin_boundaries
from national_tool_metrics.outputs import write_card_output
import national_tool_metrics.boundaries as boundaries_module
import national_tool_metrics.sections.adaptation_outcomes as outcomes_section

importlib.reload(boundaries_module)
importlib.reload(outcomes_section)
from national_tool_metrics.boundaries import load_admin_boundaries
from national_tool_metrics.sections.adaptation_outcomes import (
    ADAPTATION_OUTCOMES_CARD_DIMENSIONS,
    ADAPTATION_OUTCOMES_CARD_OPTIONAL_DIMENSIONS,
    DRY_PROOFING_CARD,
    FLOOD_PROTECTION_CARD,
    FLOOD_PROTECTION_RETURN_PERIODS,
    RELOCATION_CARD,
    URBANISATION_THRESHOLDS,
    assemble_adaptation_outcomes_card_metrics,
    build_dry_proofing_metrics,
    build_flood_protection_metrics,
    build_relocation_metrics,
)

In [ ]:
config = load_country_config("MOZ", repo_root=REPO_ROOT)
admin_regions = load_admin_boundaries(config)

print(f"Country: {config.country.name} ({config.country.iso3})")
print(f"Administrative level: {config.country.admin_level.upper()}")
print(f"Administrative regions: {len(admin_regions):,}")
print(f"Source directory: {config.source('adaptation_outcomes_dir')}")

## 1. Calculate Dry-proofing Outcomes

Retain dry-proofed area in square metres and calculate avoided values as baseline minus adapted.

In [ ]:
dry_proofing_metrics = build_dry_proofing_metrics(config, admin_regions)
print(f"Calculated {len(dry_proofing_metrics):,} dry-proofing rows")
dry_proofing_metrics.head()

## 2. Calculate Relocation Outcomes

Calculate capital stock relocated and benefits for all seven degree-of-urbanisation thresholds.

In [ ]:
relocation_metrics = build_relocation_metrics(config, admin_regions)
print(f"Calculated {len(relocation_metrics):,} relocation rows")
relocation_metrics[[
    "adm_id",
    "urbanisation_threshold_code",
    "urbanisation_threshold",
    "cost_capstock_relocated",
    "economic_avoided_total",
    "social_avoided_total",
    "social_change_ci",
]].head(7)

## 3. Calculate Flood-protection Outcomes

Calculate costs and benefits for all 35 combinations of seven degree-of-urbanisation thresholds and five design return periods. Jointly missing cost estimates remain blank rather than being converted to zero.

In [ ]:
flood_protection_metrics = build_flood_protection_metrics(config, admin_regions)
print(f"Calculated {len(flood_protection_metrics):,} flood-protection rows")
flood_protection_metrics[[
    "adm_id",
    "urbanisation_threshold_code",
    "urbanisation_threshold",
    "design_return_period_years",
    "cost_adaptation_cost",
    "cost_min_adaptation_cost",
    "cost_max_adaptation_cost",
    "economic_avoided_total",
    "social_avoided_total",
    "social_change_ci",
]].head(10)

## 4. Review Calculations

Inspect retained negative avoided values and missing flood-protection cost estimates before reshaping.

In [ ]:
for name, frame in {
    DRY_PROOFING_CARD: dry_proofing_metrics,
    RELOCATION_CARD: relocation_metrics,
    FLOOD_PROTECTION_CARD: flood_protection_metrics,
}.items():
    avoided_columns = [column for column in frame if "_avoided_" in column]
    negative_count = int((frame[avoided_columns] < 0).sum().sum())
    print(f"{name}: {negative_count:,} retained negative avoided values")

cost_columns = [
    "cost_adaptation_cost",
    "cost_min_adaptation_cost",
    "cost_max_adaptation_cost",
]
print(
    "Flood-protection scenarios with no cost estimate:",
    int(flood_protection_metrics[cost_columns].isna().all(axis=1).sum()),
)

## 5. Assemble Card Tables

Dry Proofing has 34 rows per administrative area, Relocation has 238, and Flood Protection has 1,260.

In [ ]:
card_metrics = assemble_adaptation_outcomes_card_metrics(
    config,
    admin_regions,
    dry_proofing_metrics,
    relocation_metrics,
    flood_protection_metrics,
)
dry_proofing_card = card_metrics[DRY_PROOFING_CARD]
relocation_card = card_metrics[RELOCATION_CARD]
flood_protection_card = card_metrics[FLOOD_PROTECTION_CARD]

for card, frame in card_metrics.items():
    print(f"{card}: {len(frame):,} rows")
flood_protection_card.head(10)

In [ ]:
assert len(dry_proofing_card) == len(admin_regions) * 34
assert len(relocation_card) == len(admin_regions) * 34 * len(URBANISATION_THRESHOLDS)
assert len(flood_protection_card) == (
    len(admin_regions)
    * 36
    * len(URBANISATION_THRESHOLDS)
    * len(FLOOD_PROTECTION_RETURN_PERIODS)
)
assert set(flood_protection_card["design_return_period_years"]) == set(
    FLOOD_PROTECTION_RETURN_PERIODS
)
flood_cost = flood_protection_card[
    flood_protection_card["metric"] == "adaptation_cost"
]
assert set(flood_cost["statistic"]) == {"estimate", "lower", "upper"}
assert set(flood_cost["unit"]) == {"million_usd"}
for card in card_metrics.values():
    assert "reduction_percent" not in set(card["statistic"])
    assert "quintile_ratio" not in set(card["metric"])

flood_protection_card.groupby(
    ["urbanisation_threshold_code", "design_return_period_years"],
).size()

## 6. Export

Write the three validated Adaptation Outcomes card CSVs.

In [ ]:
output_paths = {}
for card, frame in card_metrics.items():
    output_paths[card] = write_card_output(
        frame,
        config,
        section="adaptation_outcomes",
        card=card,
        dimension_columns=ADAPTATION_OUTCOMES_CARD_DIMENSIONS[card],
        optional_dimension_columns=(
            ADAPTATION_OUTCOMES_CARD_OPTIONAL_DIMENSIONS[card]
        ),
    )
    print(f"Exported {card} metrics to: {output_paths[card]}")